In [99]:
import numpy as np
import pandas as pds
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [100]:
from alpha_vantage.timeseries import TimeSeries

# api_key = 'YOUR_ALPHA_VANTAGE_API_KEY'
api_key = 'SSW9AMG17A9Y5BBA'

In [101]:
ts = TimeSeries(key=api_key, output_format='pandas')

In [102]:
# NOTE: the free Alpha Vantage tier no longer covers what this cell used:
#   - get_daily_adjusted(...)            -> premium endpoint entirely
#   - get_daily(..., outputsize='full')  -> 'full' is a premium parameter too
# Free options that still work (verified 2026-09):
#   - get_daily('AMZN', outputsize='compact') -> last 100 trading days
#   - get_monthly('AMZN')                     -> full history since 1999, monthly bars
# amzn, _ = ts.get_monthly('AMZN')  # uncomment for full history, monthly bars

amzn, _ = ts.get_daily(symbol='AMZN', outputsize='compact')  # last ~100 trading days
amzn

,1. open,2. high,3. low,4. close,5. volume
date,,,,,
2026-09-29,246.990,248.8389,245.140,246.67,33638566.0
2026-09-28,246.625,247.7699,244.725,246.15,32596709.0
2026-09-25,248.560,250.8775,247.180,249.67,34263902.0
2026-09-24,246.020,250.4300,245.600,249.38,33040052.0
2026-09-23,253.000,253.9900,247.760,249.27,44209157.0
...,...,...,...,...,...
2026-05-13,264.430,270.7200,263.200,270.13,39670915.0
2026-05-12,266.710,267.6500,262.630,265.82,36776221.0
2026-05-11,269.835,273.6300,268.480,268.99,38176696.0


In [103]:
amzn.columns = ['OPEN AMZN', 'HIGH AMZN', 'LOW AMZN', 'CLOSE AMZN', 'VOLUME AMZN']
amzn 

,OPEN AMZN,HIGH AMZN,LOW AMZN,CLOSE AMZN,VOLUME AMZN
date,,,,,
2026-09-29,246.990,248.8389,245.140,246.67,33638566.0
2026-09-28,246.625,247.7699,244.725,246.15,32596709.0
2026-09-25,248.560,250.8775,247.180,249.67,34263902.0
2026-09-24,246.020,250.4300,245.600,249.38,33040052.0
2026-09-23,253.000,253.9900,247.760,249.27,44209157.0
...,...,...,...,...,...
2026-05-13,264.430,270.7200,263.200,270.13,39670915.0
2026-05-12,266.710,267.6500,262.630,265.82,36776221.0
2026-05-11,269.835,273.6300,268.480,268.99,38176696.0


In [104]:
amzn = amzn.sort_index(axis=0, ascending=True)
pctamzn = amzn.pct_change().dropna()
pctamzn

,OPEN AMZN,HIGH AMZN,LOW AMZN,CLOSE AMZN,VOLUME AMZN
date,,,,,
2026-05-08,-0.012326,-0.009507,-0.001996,0.005568,-0.032324
2026-05-11,-0.006608,-0.001350,-0.005445,-0.013532,0.099158
2026-05-12,-0.011581,-0.021854,-0.021789,-0.011785,-0.036684
2026-05-13,-0.008549,0.011470,0.002170,0.016214,0.078711
2026-05-14,0.017850,0.000221,0.013032,-0.010773,-0.210040
...,...,...,...,...,...
2026-09-23,-0.008271,-0.019344,-0.023721,-0.022394,0.053036
2026-09-24,-0.027589,-0.014016,-0.008718,0.000441,-0.252642
2026-09-25,0.010324,0.001787,0.006433,0.001163,0.037041


### Set of data

In [105]:
# a must fit the actual data length: compact daily data only has ~100 rows
l = len(amzn)
a = int(l * 0.9)  # first 90% to train on, last 10% to visualize

X_train = amzn.iloc[0:a-1, 0:].values
y_train = amzn[["CLOSE AMZN"]].iloc[1:a:].values

X_visu = amzn.iloc[a:l-1:].values
y_visu = amzn[["CLOSE AMZN"]].iloc[a+1:l,:].values

print(f"{l} rows total -> X_train: {X_train.shape}, y_train: {y_train.shape}, X_visu: {X_visu.shape}, y_visu: {y_visu.shape}")

amzn

100 rows total -> X_train: (89, 5), y_train: (89, 1), X_visu: (9, 5), y_visu: (9, 1)


,OPEN AMZN,HIGH AMZN,LOW AMZN,CLOSE AMZN,VOLUME AMZN
date,,,,,
2026-05-07,275.020,276.6300,270.490,271.17,35892879.0
2026-05-08,271.630,274.0000,269.950,272.68,34732681.0
2026-05-11,269.835,273.6300,268.480,268.99,38176696.0
2026-05-12,266.710,267.6500,262.630,265.82,36776221.0
2026-05-13,264.430,270.7200,263.200,270.13,39670915.0
...,...,...,...,...,...
2026-09-23,253.000,253.9900,247.760,249.27,44209157.0
2026-09-24,246.020,250.4300,245.600,249.38,33040052.0
2026-09-25,248.560,250.8775,247.180,249.67,34263902.0


### Linear Regression

In [106]:
from sklearn.linear_model import LinearRegression

In [107]:
lr = LinearRegression()

In [118]:
# lr.fit(X_train, y_train)
print(lr.fit(X_train, y_train))

LinearRegression()


In [122]:
y_pred = lr.predict(X_visu)
y_pred
# print(y_pred)

array([[254.63953695],
       [254.79901002],
       [253.65717838],
       [254.14324322],
       [254.19536931],
       [254.07893784],
       [254.66298631],
       [254.59898945],
       [254.68616935]])

### Visualisation

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot()

<Figure size 1200x600 with 0 Axes>

<Figure size 1200x600 with 0 Axes>